# Five-modality factor-model benchmark — Haniffa B cells

Runs the six-model factor benchmark (SemanticSCVI-Geneformer/GenePT, LDVAE+, scHPF,
cNMF, EXPIMAP) on `haniffa_b_clean.h5ad` and compares them along five interpretable axes,
with all logic in `benchmark_modalities.py`:

1. **Gene sparsity** — few important genes per factor (good) vs all genes loaded.
2. **Program independence** — pairwise factor collinearity (some allowed, not too much).
3. **Stability** — retrain on fewer cells; the recovered representation should recur
   (rotation-invariant CCA on shared cells; legacy per-axis |corr| kept for reference).
4. **MSigDB best-program-per-factor** — fraction of factors whose best-fitting program is
   BH-significant **and** large-effect.
5. **Factor↔metadata** — factors should track biology, not batch.

Same pipeline as `benchmark_modalities_haniffa_cd8.ipynb`; **only the Cell-1 config**
(data paths, gene-set library, batch/metadata columns, cache dirs) differs. Models are
trained fresh into `.model_cache_haniffa_b` on first run, then loaded.

Output: self-contained `benchmark_results/haniffa_b_modalities/report_modalities.html`.


In [ ]:
# ============================================================
# Config — paths, hyperparameters (mirror the reference notebook), modality knobs.
# ============================================================
import hashlib, json
from pathlib import Path


def _find_nb_dir() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        for cand in (
            p / "notebooks" / "benchmark_helpers.py",
            p / "benchmark_helpers.py",
            p / "scvi-tools-neural-nmf" / "notebooks" / "benchmark_helpers.py",
        ):
            if cand.exists():
                return cand.parent.resolve()
    raise FileNotFoundError(
        f"benchmark_helpers.py not found from {Path.cwd()}. "
        "Launch jupyter under the scvi-tools-neural-nmf tree, or set NB_DIR manually."
    )


def _slug(kwargs, max_epochs, warmup, kl_warmup, hvg, n=10):
    blob = json.dumps({"kwargs": dict(sorted(kwargs.items())), "max_epochs": max_epochs,
                       "warmup_epochs": warmup, "n_epochs_kl_warmup": kl_warmup,
                       "hvg_top_n": hvg}, default=str, sort_keys=True)
    return hashlib.sha1(blob.encode()).hexdigest()[:n]


NB_DIR = _find_nb_dir()

# ====== PER-NOTEBOOK CONFIG (only this block differs across the modalities notebooks) ======
RUN_NAME     = "haniffa_b"        # -> benchmark_results/<RUN_NAME>_modalities, .model_cache_<RUN_NAME>
DATASET_STEM = "haniffa_b_clean"    # <stem>.h5ad + <stem>_{geneformer,genept_3large}.pt in NB_DIR
LIB2_NAME    = "lib2_bcell"       # NB_DIR/<LIB2_NAME>.gmt  (lib1_immune is universal)
BATCH_KEY    = 'Site'          # batch / condition key for the VAEs + EXPIMAP
BIO_VARS     = ['cell_type', 'Status', 'Status_on_day_collection_summary', 'Sex']           # mod5 biological obs columns (higher assoc = better)
BATCH_VARS   = ['Site', 'donor_id']         # mod5 technical/batch obs columns (lower assoc = better)
# ===========================================================================================

ADATA_PATH = NB_DIR / f"{DATASET_STEM}.h5ad"
SEMANTIC_CACHE_GENEFORMER = NB_DIR / f"{DATASET_STEM}_geneformer.pt"
SEMANTIC_CACHE_GENEPT = NB_DIR / f"{DATASET_STEM}_genept_3large.pt"
GENEPT_PICKLE_PATH = NB_DIR / "GenePT_gene_protein_embedding_model_3_text.pickle"
LIB1_GMT = NB_DIR / "lib1_immune.gmt"
LIB2_GMT = NB_DIR / f"{LIB2_NAME}.gmt"
OUT_DIR = NB_DIR / "benchmark_results" / f"{RUN_NAME}_modalities"
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_DIR = NB_DIR / f".model_cache_{RUN_NAME}"          # per-run model cache (fresh train)
STABILITY_DIR = MODEL_CACHE_DIR / "stability"                  # subsample retrains

# ---- Preprocessing / shared ----
HVG_TOP_N = 2500
HVG_FLAVOR = "seurat_v3"
SUBSAMPLE_N = 40000
N_LATENT = 10
GENE_MAPPING = ("feature_id", "feature_name")
MODEL_NAMES = ["semantic_geom", "semantic_genept", "ldvae_nn", "schpf_k10", "cnmf_k10", "expimap_k10"]

# ---- SemanticSCVI (Geneformer / GenePT) ----
SEM_GEOM_MAX_EPOCHS = SEM_GENEPT_MAX_EPOCHS = 200
SEM_GEOM_WARMUP = SEM_GENEPT_WARMUP = 20
SEM_GEOM_KL_WARMUP = SEM_GENEPT_KL_WARMUP = 100
_SEM_KWARGS = dict(loss_mode="geometric", coherence_weight=2000.0, n_gene_sample=1024,
                   n_latent=N_LATENT, n_layers=2, n_hidden=128, dropout_rate=0.1,
                   gene_likelihood="nb", weights_positive=True, use_batch_norm=False)
SEM_GEOM_KWARGS = dict(_SEM_KWARGS)
SEM_GENEPT_KWARGS = dict(_SEM_KWARGS)

# ---- LDVAE+ ----
LDVAE_MAX_EPOCHS = 250
LDVAE_KWARGS = dict(n_hidden=128, n_latent=N_LATENT, n_layers=1, dropout_rate=0.1,
                    gene_likelihood="nb", weights_positive=True, use_batch_norm=False)

# ---- scHPF ----
N_FACTORS = N_LATENT

# ---- cNMF (consensus NMF, Kotliar et al. 2019) ----
CNMF_N_ITER = 20                  # NMF replicates pooled into the consensus
CNMF_DENSITY_THRESHOLD = 0.5      # consensus outlier-spectra filter
CNMF_NUM_HIGHVAR_GENES = None     # None -> force cNMF onto ALL pipeline genes (genes_file)
CNMF_BETA_LOSS = "frobenius"
CNMF_INIT = "random"
CNMF_LOADINGS = "score"           # "score" (z-scored, canonical) or "tpm" (non-negative)

# ---- EXPIMAP ----
EXPIMAP_MASK_TERMS = [
    "HALLMARK_INTERFERON_GAMMA_RESPONSE", "HALLMARK_IL2_STAT5_SIGNALING",
    "HALLMARK_TNFA_SIGNALING_VIA_NFKB", "HALLMARK_INFLAMMATORY_RESPONSE",
    "HALLMARK_OXIDATIVE_PHOSPHORYLATION", "HALLMARK_GLYCOLYSIS",
    "HALLMARK_MTORC1_SIGNALING", "HALLMARK_HYPOXIA",
    "HALLMARK_E2F_TARGETS", "HALLMARK_G2M_CHECKPOINT",
]
EXPIMAP_SOURCE_GMT = LIB1_GMT
EXPIMAP_MASK_GMT = NB_DIR / f"{ADATA_PATH.stem}_expimap_hallmark.gmt"
EXPIMAP_N_EPOCHS = 400
EXPIMAP_ALPHA = 0.7
EXPIMAP_ALPHA_KL = 0.5
EXPIMAP_ALPHA_EPOCH_ANNEAL = 130
EXPIMAP_KWARGS = dict(condition_key=BATCH_KEY, recon_loss="nb", hidden_layer_sizes=(256, 256))

# ---- Param-hash cache dirs (so a re-run LOADS instead of retraining) ----
SEM_GEOM_CACHE = MODEL_CACHE_DIR / "semantic_scvi" / _slug(
    {**SEM_GEOM_KWARGS, "batch_key": BATCH_KEY}, SEM_GEOM_MAX_EPOCHS, SEM_GEOM_WARMUP, SEM_GEOM_KL_WARMUP, HVG_TOP_N)
SEM_GENEPT_CACHE = MODEL_CACHE_DIR / "semantic_scvi_genept" / _slug(
    {**SEM_GENEPT_KWARGS, "batch_key": BATCH_KEY}, SEM_GENEPT_MAX_EPOCHS, SEM_GENEPT_WARMUP, SEM_GENEPT_KL_WARMUP, HVG_TOP_N)
LDVAE_CACHE = MODEL_CACHE_DIR / f"ldvae_nn_batch_{BATCH_KEY or 'none'}"
EXPIMAP_CACHE = MODEL_CACHE_DIR / "expimap_k10" / _slug(
    {**EXPIMAP_KWARGS, "terms": tuple(EXPIMAP_MASK_TERMS), "alpha": EXPIMAP_ALPHA,
     "alpha_kl": EXPIMAP_ALPHA_KL, "alpha_epoch_anneal": EXPIMAP_ALPHA_EPOCH_ANNEAL},
    EXPIMAP_N_EPOCHS, 0, EXPIMAP_ALPHA_EPOCH_ANNEAL, HVG_TOP_N)

# ---- cfg dict consumed by benchmark_modalities.train_all_models (stability retrains) ----
cfg = dict(
    N_LATENT=N_LATENT, BATCH_KEY=BATCH_KEY, N_FACTORS=N_FACTORS,
    CNMF_N_ITER=CNMF_N_ITER, CNMF_DENSITY_THRESHOLD=CNMF_DENSITY_THRESHOLD,
    CNMF_NUM_HIGHVAR_GENES=CNMF_NUM_HIGHVAR_GENES, CNMF_BETA_LOSS=CNMF_BETA_LOSS,
    CNMF_INIT=CNMF_INIT, CNMF_LOADINGS=CNMF_LOADINGS,
    SEM_GEOM_MAX_EPOCHS=SEM_GEOM_MAX_EPOCHS, SEM_GEOM_WARMUP=SEM_GEOM_WARMUP, SEM_GEOM_KL_WARMUP=SEM_GEOM_KL_WARMUP, SEM_GEOM_KWARGS=SEM_GEOM_KWARGS,
    SEM_GENEPT_MAX_EPOCHS=SEM_GENEPT_MAX_EPOCHS, SEM_GENEPT_WARMUP=SEM_GENEPT_WARMUP, SEM_GENEPT_KL_WARMUP=SEM_GENEPT_KL_WARMUP, SEM_GENEPT_KWARGS=SEM_GENEPT_KWARGS,
    LDVAE_MAX_EPOCHS=LDVAE_MAX_EPOCHS, LDVAE_KWARGS=LDVAE_KWARGS,
    EXPIMAP_MASK_GMT=str(EXPIMAP_MASK_GMT), EXPIMAP_SOURCE_GMT=str(EXPIMAP_SOURCE_GMT), EXPIMAP_MASK_TERMS=EXPIMAP_MASK_TERMS,
    EXPIMAP_N_EPOCHS=EXPIMAP_N_EPOCHS, EXPIMAP_ALPHA=EXPIMAP_ALPHA, EXPIMAP_ALPHA_KL=EXPIMAP_ALPHA_KL,
    EXPIMAP_ALPHA_EPOCH_ANNEAL=EXPIMAP_ALPHA_EPOCH_ANNEAL, EXPIMAP_KWARGS=EXPIMAP_KWARGS,
)

# ---- Force-retrain switches (full models) ----
FORCE_TRAIN = dict(geom=False, genept=False, ldvae=False, schpf=False, cnmf=False, expimap=False)

# ---- Modality knobs ----
SPARSITY_MASS_FRAC = 0.9     # mod1: #genes to reach this fraction of loading mass
SPARSITY_EPS = 1e-9          # mod1: |loading|>eps counts as "used" (structural-zero robust)
COLLIN_THRESHOLD = 0.5       # mod2: |r| above this counts as a redundant program pair
N_TOP = 30                   # mod4: top genes per factor for enrichment
Q_THRESH = 0.05              # mod4: BH-adjusted p threshold
ER_THRESH = 2.0              # mod4: fold-enrichment ("large effect") threshold
BEST_BY = "pval"             # mod4: best program per factor by "pval" (min p) or "ER" (max fold)
FRACTIONS = [0.75, 0.8, 0.9]  # mod3: cell subsample fractions
SEEDS = [0, 1]               # mod3: seeds per fraction

# mod4 held-out eval: drop EXPIMAP's mask programs from H/C2 (applied to ALL models) so its
# enrichment isn't circular — its factors ARE those HALLMARK terms. C7 is mask-disjoint.
if EXPIMAP_MASK_GMT.exists():
    MOD4_HELD_OUT_TERMS = [ln.split("\t")[0] for ln in EXPIMAP_MASK_GMT.read_text().splitlines() if ln.strip()]
else:
    MOD4_HELD_OUT_TERMS = list(EXPIMAP_MASK_TERMS)

print(f"NB_DIR = {NB_DIR}")
print(f"RUN_NAME = {RUN_NAME} | ADATA = {ADATA_PATH.name} | LIB2 = {LIB2_GMT.name} | BATCH_KEY = {BATCH_KEY}")
print(f"OUT_DIR = {OUT_DIR}")
print(f"mod4 held-out terms ({len(MOD4_HELD_OUT_TERMS)}): {MOD4_HELD_OUT_TERMS}")


In [ ]:
import sys
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import importlib
import numpy as np
import pandas as pd
import scanpy as sc
import torch

import anndata as _ad
if not hasattr(_ad, "read"):
    _ad.read = _ad.read_h5ad  # scArches 0.6.1 shim

import benchmark_modalities as bm
importlib.reload(bm)
from benchmark_helpers import (
    NMFWrapper, _ExpimapAdapter, _ScviAdapter, build_expimap_mask_gmt,
    get_or_build_geneformer_map, get_or_build_genept_map,
    train_or_load_expimap, train_or_load_nonneg_ldvae, train_or_load_pickle,
    train_or_load_semantic_scvi,
)
from train_schpf import train_schpf_model
from train_cnmf import train_cnmf_model

print(f"CUDA available: {torch.cuda.is_available()}")


In [ ]:
# Load adata, build/cache semantic maps, HVG-subset (mirrors the reference notebook).
adata = sc.read_h5ad(ADATA_PATH)
print("adata (raw):", adata.shape)

semantic_map_geneformer = get_or_build_geneformer_map(adata, SEMANTIC_CACHE_GENEFORMER)
semantic_map_genept = get_or_build_genept_map(adata, SEMANTIC_CACHE_GENEPT, GENEPT_PICKLE_PATH)

if HVG_TOP_N is not None:
    sc.pp.highly_variable_genes(adata, n_top_genes=HVG_TOP_N, flavor=HVG_FLAVOR, subset=False)
    keep = adata.var["highly_variable"].values
    adata = adata[:, keep].copy()
    keep_t = torch.as_tensor(keep)
    semantic_map_geneformer = semantic_map_geneformer[keep_t]
    semantic_map_genept = semantic_map_genept[keep_t]

if SUBSAMPLE_N is not None and adata.n_obs > SUBSAMPLE_N:
    sc.pp.subsample(adata, n_obs=SUBSAMPLE_N, random_state=42)

maps = {"geneformer": semantic_map_geneformer, "genept": semantic_map_genept}
print("adata (HVG/subsampled):", adata.shape)
print("geneformer map:", tuple(semantic_map_geneformer.shape), "| genept map:", tuple(semantic_map_genept.shape))


In [ ]:
# Full-data models — load from the reference cache (retrain only if FORCE_TRAIN flips).
adata_geom = adata.copy()
m_geom = train_or_load_semantic_scvi(
    adata_geom, semantic_map_geneformer, cache_dir=SEM_GEOM_CACHE, force_train=FORCE_TRAIN["geom"],
    max_epochs=SEM_GEOM_MAX_EPOCHS, warmup_epochs=SEM_GEOM_WARMUP,
    n_epochs_kl_warmup=SEM_GEOM_KL_WARMUP, batch_key=BATCH_KEY, **SEM_GEOM_KWARGS)

adata_genept = adata.copy()
m_genept = train_or_load_semantic_scvi(
    adata_genept, semantic_map_genept, cache_dir=SEM_GENEPT_CACHE, force_train=FORCE_TRAIN["genept"],
    max_epochs=SEM_GENEPT_MAX_EPOCHS, warmup_epochs=SEM_GENEPT_WARMUP,
    n_epochs_kl_warmup=SEM_GENEPT_KL_WARMUP, batch_key=BATCH_KEY, **SEM_GENEPT_KWARGS)

adata_ldvae = adata.copy()
m_ldvae = train_or_load_nonneg_ldvae(
    adata_ldvae, cache_dir=LDVAE_CACHE, force_train=FORCE_TRAIN["ldvae"],
    max_epochs=LDVAE_MAX_EPOCHS, batch_key=BATCH_KEY, **LDVAE_KWARGS)

m_schpf = train_or_load_pickle(
    "scHPF", lambda: train_schpf_model(adata, n_factors=N_FACTORS),
    cache_path=MODEL_CACHE_DIR / "schpf_k10.pkl", force_train=FORCE_TRAIN["schpf"])

# cNMF (consensus NMF) — runs prepare/factorize/combine/consensus on CPU (~15-25 min),
# then caches the lightweight wrapper to cnmf_k10.pkl (subsequent loads are instant).
m_cnmf = train_or_load_pickle(
    "cNMF",
    lambda: train_cnmf_model(
        adata, n_factors=N_FACTORS, output_dir=MODEL_CACHE_DIR / "cnmf_k10_run",
        name="cnmf_k10", n_iter=CNMF_N_ITER, density_threshold=CNMF_DENSITY_THRESHOLD,
        num_highvar_genes=CNMF_NUM_HIGHVAR_GENES, seed=42,
        beta_loss=CNMF_BETA_LOSS, init=CNMF_INIT, loadings=CNMF_LOADINGS),
    cache_path=MODEL_CACHE_DIR / "cnmf_k10.pkl", force_train=FORCE_TRAIN["cnmf"])

if not EXPIMAP_MASK_GMT.exists():
    build_expimap_mask_gmt(adata, EXPIMAP_SOURCE_GMT, EXPIMAP_MASK_TERMS, EXPIMAP_MASK_GMT)
adata_expimap = adata.copy()
m_expimap = train_or_load_expimap(
    adata_expimap, gmt_path=EXPIMAP_MASK_GMT, cache_dir=EXPIMAP_CACHE, force_train=FORCE_TRAIN["expimap"],
    n_epochs=EXPIMAP_N_EPOCHS, alpha_kl=EXPIMAP_ALPHA_KL, alpha=EXPIMAP_ALPHA,
    alpha_epoch_anneal=EXPIMAP_ALPHA_EPOCH_ANNEAL, **EXPIMAP_KWARGS)

models = {
    "semantic_geom": _ScviAdapter(m_geom, adata_geom),
    "semantic_genept": _ScviAdapter(m_genept, adata_genept),
    "ldvae_nn": _ScviAdapter(m_ldvae, adata_ldvae),
    "schpf_k10": m_schpf,
    "cnmf_k10": m_cnmf,
    "expimap_k10": _ExpimapAdapter(m_expimap, adata_expimap),
}
loadings = bm.get_loadings_dict(models)
latents = bm.get_latent_dict(models)
for name in MODEL_NAMES:
    print(f"{name}: loadings {loadings[name].shape}, latent {latents[name].shape}")


In [ ]:
# Modality 1 — gene sparsity (structural-zero-robust: within-support Gini + fraction used)
sparsity_csv = OUT_DIR / "mod1_sparsity.csv"
_need = {"gini_support", "frac_used"}
if sparsity_csv.exists() and _need.issubset(pd.read_csv(sparsity_csv, nrows=0).columns):
    sparsity_df = pd.read_csv(sparsity_csv)
else:
    sparsity_df = bm.compute_sparsity(loadings, mass_frac=SPARSITY_MASS_FRAC, eps=SPARSITY_EPS)
    sparsity_df.to_csv(sparsity_csv, index=False)
sparsity_summary = (sparsity_df.groupby("Model")[["gini_support", "frac_used", "eff_genes", "genes_to_mass"]]
                    .median().reindex(MODEL_NAMES).reset_index())
bm.plot_sparsity(sparsity_df, OUT_DIR / "mod1_sparsity.png", MODEL_NAMES, mass_frac=SPARSITY_MASS_FRAC)
sparsity_summary


In [ ]:
# Modality 2 — program independence / collinearity
collin_summary, collin_mats = bm.compute_collinearity(latents, threshold=COLLIN_THRESHOLD)
collin_summary = collin_summary.set_index("Model").reindex(MODEL_NAMES).reset_index()
collin_summary.to_csv(OUT_DIR / "mod2_collinearity.csv", index=False)
bm.plot_collinearity(collin_mats, collin_summary, OUT_DIR / "mod2_collinearity.png", MODEL_NAMES, threshold=COLLIN_THRESHOLD)
collin_summary


In [ ]:
# Modality 4 — MSigDB best-program-per-factor (BH-corrected per model x library)
# Held-out: EXPIMAP's mask programs dropped from H/C2 for ALL models (its factors ARE those
# HALLMARK terms). C7 is mask-disjoint. 4-panel view: ER histogram, -log10(q) histogram,
# %sig+strong (q & ER cutoff), %sig (q cutoff only).
libraries = bm.load_libraries(str(LIB1_GMT) if LIB1_GMT.exists() else None,
                              str(LIB2_GMT) if LIB2_GMT.exists() else None,
                              exclude_terms=MOD4_HELD_OUT_TERMS)
for k, v in libraries.items():
    print(f"  {k}: {len(v)} programs")
bp_detail, bp_summary = bm.best_program_proportions(
    loadings, libraries, adata, GENE_MAPPING,
    n_top=N_TOP, q_thresh=Q_THRESH, er_thresh=ER_THRESH, best_by=BEST_BY)
bp_detail.to_csv(OUT_DIR / "mod4_best_program_detail.csv", index=False)
bp_summary.to_csv(OUT_DIR / "mod4_best_program_summary.csv", index=False)
bm.plot_best_program(bp_detail, OUT_DIR / "mod4_best_program.png", MODEL_NAMES, Q_THRESH, ER_THRESH)
bp_summary.pivot(index="Model", columns="Library", values="prop_sig_strong").reindex(MODEL_NAMES)


In [ ]:
# Modality 5 — factor <-> metadata association (max eta^2 per variable)
meta_detail, meta_summary = bm.compute_metadata_assoc(latents, adata.obs, BIO_VARS, BATCH_VARS)
meta_detail.to_csv(OUT_DIR / "mod5_metadata_detail.csv", index=False)
meta_summary = meta_summary.set_index("Model").reindex(MODEL_NAMES).reset_index()
meta_summary.to_csv(OUT_DIR / "mod5_metadata_summary.csv", index=False)
bm.plot_metadata(meta_detail, OUT_DIR / "mod5_metadata.png", MODEL_NAMES)
meta_summary


In [ ]:
# Modality 3 — stability under cell subsampling (cached per frac x seed).
# Retrains all six models on each subsample (loads from STABILITY_DIR cache if present),
# then scores two ways: rotation-invariant cell-embedding CCA on the shared cells, and
# the legacy per-axis Hungarian |corr|. The per-axis metric unfairly penalizes rotationally
# non-identifiable VAE latents; the embedding CCA measures whether the same representation
# recurs regardless of basis.
stability_csv = OUT_DIR / "mod3_stability.csv"
_need = {"embed_corr", "matched_corr"}
if stability_csv.exists():
    _cached = pd.read_csv(stability_csv)
    _fresh = (_need.issubset(_cached.columns)
              and set(FRACTIONS).issubset(set(_cached["fraction"].unique())))
else:
    _cached, _fresh = None, False
if _fresh:
    stability_df = _cached[_cached["fraction"].isin(FRACTIONS)].copy()
else:
    full_latent = {n: pd.DataFrame(Z, index=adata.obs_names) for n, Z in latents.items()}
    # reuse already-scored (fraction, seed) rows for the kept fractions; compute the rest
    if _cached is not None and _need.issubset(_cached.columns):
        _keep = _cached[_cached["fraction"].isin(FRACTIONS)]
        rows = _keep[["Model", "fraction", "seed", *sorted(_need)]].to_dict("records")
    else:
        rows = []
    done = {(r["fraction"], r["seed"]) for r in rows}
    for frac in FRACTIONS:
        for seed in SEEDS:
            if (frac, seed) in done:
                continue
            print(f"\n=== stability: fraction={frac} seed={seed} ===")
            sub = adata.copy()
            sc.pp.subsample(sub, fraction=frac, random_state=seed)
            run_dir = STABILITY_DIR / f"frac{frac}_seed{seed}"
            sub_models = bm.train_all_models(sub, maps, cfg, run_dir, seed=seed)
            sub_loadings = bm.get_loadings_dict(sub_models)
            sub_latent = {n: pd.DataFrame(Z, index=sub.obs_names)
                          for n, Z in bm.get_latent_dict(sub_models).items()}
            for name in MODEL_NAMES:
                matched = bm.match_factors(loadings[name], sub_loadings[name])[0]
                embed = bm.embedding_stability(full_latent[name], sub_latent[name])
                rows.append(dict(Model=name, fraction=frac, seed=seed,
                                 matched_corr=matched, embed_corr=embed))
                print(f"  {name}: embed={embed:.3f} matched={matched:.3f}")
    stability_df = pd.DataFrame(rows)
    stability_df.to_csv(stability_csv, index=False)
bm.plot_stability(stability_df, OUT_DIR / "mod3_stability.png", MODEL_NAMES)
stability_summary = stability_df.groupby(["Model", "fraction"])["embed_corr"].mean().unstack().reindex(MODEL_NAMES)
stability_summary


In [ ]:
# Leaderboard + self-contained HTML report
lb = pd.DataFrame({"Model": MODEL_NAMES}).set_index("Model")
lb["sparsity_gini_support"] = sparsity_df.groupby("Model")["gini_support"].median().reindex(MODEL_NAMES)
lb["frac_genes_used"] = sparsity_df.groupby("Model")["frac_used"].median().reindex(MODEL_NAMES)
lb["mean_abs_collin"] = collin_summary.set_index("Model")["mean_abs_offdiag"].reindex(MODEL_NAMES)
# graded modality-4: median best-match ER (significant factors), averaged over libraries
lb["bestprog_medianER"] = bp_summary.groupby("Model")["median_ER_sig"].mean().reindex(MODEL_NAMES)
lb["meta_bio_minus_batch"] = meta_summary.set_index("Model")["bio_minus_batch"].reindex(MODEL_NAMES)
# modality-3 stability at the lowest fraction: rotation-invariant cell-embedding CCA
# (representation recovery on shared cells) + legacy per-axis matched |corr| for reference.
_min_frac = min(FRACTIONS)
_stab_lf = stability_df[stability_df.fraction == _min_frac].groupby("Model")
lb["stability_embed_lowfrac"] = _stab_lf["embed_corr"].mean().reindex(MODEL_NAMES)
lb["stability_matched_lowfrac"] = _stab_lf["matched_corr"].mean().reindex(MODEL_NAMES)
lb = lb.reset_index()

knobs = dict(HVG_TOP_N=HVG_TOP_N, N_LATENT=N_LATENT, BATCH_KEY=BATCH_KEY,
             SPARSITY_MASS_FRAC=SPARSITY_MASS_FRAC, COLLIN_THRESHOLD=COLLIN_THRESHOLD,
             N_TOP=N_TOP, Q_THRESH=Q_THRESH, ER_THRESH=ER_THRESH, BEST_BY=BEST_BY,
             FRACTIONS=FRACTIONS, SEEDS=SEEDS, BIO_VARS=BIO_VARS, BATCH_VARS=BATCH_VARS,
             MOD4_HELD_OUT_TERMS=f"{len(MOD4_HELD_OUT_TERMS)} EXPIMAP mask terms removed from H/C2")
tables = dict(leaderboard=lb, sparsity=sparsity_summary, collinearity=collin_summary,
              best_program=bp_summary.pivot(index="Model", columns="Library", values="median_ER_sig").reindex(MODEL_NAMES).reset_index(),
              metadata=meta_summary, stability=stability_summary.reset_index())
notes = (f"Six models reused from {MODEL_CACHE_DIR.name}. Sparsity: within-support Gini (↑ concentrates) "
         f"+ fraction of genes used (global Gini dropped — inflated by structural zeros for masked models). "
         f"Independence: lower |r| = more orthogonal (some collinearity OK). Modality-3 stability: "
         f"rotation-invariant CCA of the cell representation (on shared cells) — fair to rotationally "
         f"non-identifiable VAE latents; legacy per-axis Hungarian |corr| kept for reference. "
         f"Modality-4: 4-panel view (best-match ER histogram, −log10 q histogram, %sig+strong, %sig), HELD-OUT "
         f"(EXPIMAP's {len(MOD4_HELD_OUT_TERMS)} mask terms removed from H/C2; C7 disjoint); leaderboard column = "
         f"median best-match ER of significant factors. EXPIMAP is annotation-supervised — its factors "
         f"ARE HALLMARK terms, so its enrichment reflects construction, not discovery. Metadata = max η² per variable.")
report = bm.build_modalities_report(OUT_DIR, MODEL_NAMES, adata.shape, knobs, tables, notes=notes)
print(f"Report written: {report}  ({report.stat().st_size/1e3:.0f} KB)")
lb
